In [0]:
customers = spark.table("biker_r_project.raw.customers")
customers.show(5)

+-----------+----------+---------+--------------+--------------------+--------------------+-------------+-----+--------+
|customer_id|first_name|last_name|         phone|               email|              street|         city|state|zip_code|
+-----------+----------+---------+--------------+--------------------+--------------------+-------------+-----+--------+
|          1|     Debra|    Burks|          NULL|debra.burks@yahoo...|   9273 Thorne Ave. | Orchard Park|   NY|   14127|
|          2|     Kasha|     Todd|          NULL|kasha.todd@yahoo.com|    910 Vine Street |     Campbell|   CA|   95008|
|          3|    Tameka|   Fisher|          NULL|tameka.fisher@aol...|769C Honey Creek ...|Redondo Beach|   CA|   90278|
|          4|     Daryl|   Spence|          NULL|daryl.spence@aol.com|     988 Pearl Lane |    Uniondale|   NY|   11553|
|          5|Charolette|     Rice|(916) 381-6003|charolette.rice@m...|      107 River Dr. |   Sacramento|   CA|   95820|
+-----------+----------+--------

In [0]:
from pyspark.sql.functions import *

customers_clean = (
    customers
    .dropDuplicates()
    .filter(col("customer_id").isNotNull())
    .withColumn("first_name", initcap(trim(col("first_name"))))
    .withColumn("last_name", initcap(trim(col("last_name"))))
)

customers_clean.write \
.mode("overwrite") \
.saveAsTable("biker_r_project.silver.customers")

In [0]:
brands = spark.table("biker_r_project.raw.brands")
brands_clean = brands.dropDuplicates()

brands_clean.write \
.mode("overwrite") \
.saveAsTable("biker_r_project.silver.brands")

In [0]:
categories = spark.table("biker_r_project.raw.categories")
categories_clean = categories.dropDuplicates()

categories_clean.write \
.mode("overwrite") \
.saveAsTable("biker_r_project.silver.categories")

In [0]:
products = spark.table("biker_r_project.raw.products")

products_clean = (
    products
    .dropDuplicates()
    .filter(col("product_id").isNotNull())
    .withColumn("product_name", trim(col("product_name")))
)

products_clean.write \
.mode("overwrite") \
.saveAsTable("biker_r_project.silver.products")

In [0]:
orders = spark.table("biker_r_project.raw.orders")
orders_clean = (
    orders
    .dropDuplicates()
    .filter(col("order_id").isNotNull())
)

orders_clean.write \
.mode("overwrite") \
.saveAsTable("biker_r_project.silver.orders")

In [0]:
order_items = spark.table("biker_r_project.raw.order_items")
order_items_clean = (
    order_items
    .dropDuplicates()
    .filter(col("order_id").isNotNull())
    .filter(col("product_id").isNotNull())
    .filter(col("quantity") > 0)
)

order_items_clean.write \
.mode("overwrite") \
.saveAsTable("biker_r_project.silver.order_items")

In [0]:
staffs = spark.table("biker_r_project.raw.staffs")
staffs_clean = (
    staffs
    .dropDuplicates()
    .withColumn("first_name", initcap(trim(col("first_name"))))
    .withColumn("last_name", initcap(trim(col("last_name"))))
)

staffs_clean.write \
.mode("overwrite") \
.saveAsTable("biker_r_project.silver.staffs")

In [0]:
stores = spark.table("biker_r_project.raw.stores")
stores_clean = (
    stores
    .dropDuplicates()
    .withColumn("store_name", trim(col("store_name")))
)

stores_clean.write \
.mode("overwrite") \
.saveAsTable("biker_r_project.silver.stores")

In [0]:
stocks = spark.table("biker_r_project.raw.stocks")
stocks_clean = (
    stocks
    .dropDuplicates()
    .filter(col("quantity") >= 0)
)

stocks_clean.write \
.mode("overwrite") \
.saveAsTable("biker_r_project.silver.stocks")

In [0]:
%sql
SHOW TABLES IN biker_r_project.silver;

database,tableName,isTemporary
silver,brands,false
silver,categories,false
silver,customers,false
silver,order_items,false
silver,orders,false
silver,products,false
silver,staffs,false
silver,stocks,false
silver,stores,false


In [0]:
customers.count()
customers_clean.count()

1445

In [0]:
products.groupBy("product_id").count().filter("count > 1").show()
customers.filter(col("customer_id").isNull()).count()

+----------+-----+
|product_id|count|
+----------+-----+
+----------+-----+



0

In [0]:
invalid_orders = orders.join(
    customers,
    orders.customer_id == customers.customer_id,
    "left_anti"
)

invalid_orders.show()

+--------+-----------+------------+----------+-------------+------------+--------+--------+
|order_id|customer_id|order_status|order_date|required_date|shipped_date|store_id|staff_id|
+--------+-----------+------------+----------+-------------+------------+--------+--------+
+--------+-----------+------------+----------+-------------+------------+--------+--------+



In [0]:
invalid_products = order_items.join(
    products,
    order_items.product_id == products.product_id,
    "left_anti"
)

invalid_products.show()

+--------+-------+----------+--------+----------+--------+
|order_id|item_id|product_id|quantity|list_price|discount|
+--------+-------+----------+--------+----------+--------+
+--------+-------+----------+--------+----------+--------+



In [0]:
product_dim = (
    products
    .join(brands, "brand_id")
    .join(categories, "category_id")
    .select(
        "product_id",
        "product_name",
        "brand_name",
        "category_name",
        "model_year",
        "list_price"
    )
)

product_dim.write.mode("overwrite")\
.saveAsTable("biker_r_project.silver.dim_product")

In [0]:
customer_dim = customers.select(
    "customer_id",
    "first_name",
    "last_name",
    "city",
    "state",
    "zip_code"
)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank, desc

window_spec = Window.partitionBy("store_name") \
                    .orderBy(desc("sales"))

customer_rank = customer_sales.withColumn(
    "rank",
    dense_rank().over(window_spec)
)

In [0]:
window_spec = Window.partitionBy("store_name") \
                    .orderBy("order_date")

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

store_sales = (
    sales_fact
    .groupBy("store_name","customer_name")
    .agg(sum("revenue").alias("sales"))
)

window_spec = Window.partitionBy("store_name") \
                    .orderBy(desc("sales"))

top_customers = store_sales.withColumn(
    "rank",
    dense_rank().over(window_spec)
)

In [0]:
window_spec = Window.orderBy(desc("sales"))

segments = customer_sales.withColumn(
    "segment",
    ntile(4).over(window_spec)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
from pyspark.sql.functions import broadcast

sales_fact = (
    orders
    .join(order_items,"order_id")
    .join(
        broadcast(products),
        "product_id"
    )
)

In [0]:
sales_fact.explain(True)

== Parsed Logical Plan ==
'Join UsingJoin(Inner, [product_id])
:- 'Join UsingJoin(Inner, [order_id])
:  :- 'UnresolvedRelation [biker_r_project, raw, orders], [], false
:  +- 'UnresolvedRelation [biker_r_project, raw, order_items], [], false
+- 'UnresolvedHint broadcast
   +- 'UnresolvedRelation [biker_r_project, raw, products], [], false

== Analyzed Logical Plan ==
product_id: bigint, order_id: bigint, customer_id: bigint, order_status: bigint, order_date: date, required_date: date, shipped_date: string, store_id: bigint, staff_id: bigint, item_id: bigint, quantity: bigint, list_price: double, discount: double, product_name: string, brand_id: bigint, category_id: bigint, model_year: bigint, list_price: double
Project [product_id#16320L, order_id#16310L, customer_id#16311L, order_status#16312L, order_date#16313, required_date#16314, shipped_date#16315, store_id#16316L, staff_id#16317L, item_id#16319L, quantity#16321L, list_price#16322, discount#16323, product_name#16325, brand_id#1632

In [0]:
store_kpis = (
    sales_fact
    .groupBy("store_name")
    .agg(
        countDistinct("order_id")
        .alias("orders"),

        countDistinct("customer_id")
        .alias("customers"),

        sum("revenue")
        .alias("revenue")
    )
)

In [0]:
from pyspark.sql.functions import *

monthly_sales = (
    spark.table("biker_r_project.gold.sales_fact")
    .groupBy(
        year("order_date").alias("year"),
        month("order_date").alias("month")
    )
    .agg(sum("revenue").alias("revenue"))
    .orderBy("year","month")
)

display(monthly_sales)

year,month,revenue
2016,1,215146.42409999995
2016,2,156112.3228
2016,3,180600.3285
2016,4,167144.0512
2016,5,205270.00909999997
2016,6,210562.12449999986
2016,7,199556.80889999992
2016,8,225657.37669999996
2016,9,273091.60970000003
2016,10,212078.08050000004


In [0]:
top_products = (
    spark.table("biker_r_project.gold.sales_fact")
    .groupBy("product_name")
    .agg(sum("revenue").alias("revenue"))
    .orderBy(desc("revenue"))
    .limit(10)
)

display(top_products)

product_name,revenue
Trek Slash 8 27.5 - 2016,555558.6111
Trek Conduit+ - 2016,389248.7025000002
Trek Fuel EX 8 29 - 2016,368472.72939999995
Surly Straggler 650b - 2016,226765.5509999999
Trek Domane SLR 6 Disc - 2017,211584.61529999998
Surly Straggler - 2016,203507.62000000026
Trek Remedy 29 Carbon Frameset - 2016,203380.87009999997
Trek Powerfly 8 FS Plus - 2017,188249.62349999996
Trek Madone 9.2 - 2017,175899.64819999994
Trek Silque SLR 8 Women's - 2017,174524.73150000002


Databricks visualization. Run in Databricks to view.

In [0]:
category_sales = (
    spark.table("biker_r_project.gold.sales_fact")
    .groupBy("category_name")
    .agg(sum("revenue").alias("revenue"))
    .orderBy(desc("revenue"))
)

display(category_sales)

category_name,revenue
Mountain Bikes,2715079.533699997
Road Bikes,1665098.487999998
Cruisers Bicycles,995032.6237000015
Electric Bikes,916684.7800000007
Cyclocross Bicycles,711011.8359000003
Comfort Bicycles,394020.09810000093
Children Bicycles,292189.1982000012


Databricks visualization. Run in Databricks to view.

In [0]:
store_sales = (
    spark.table("biker_r_project.gold.sales_fact")
    .groupBy("store_name")
    .agg(sum("revenue").alias("revenue"))
)

display(store_sales)

store_name,revenue
Santa Cruz Bikes,1605823.0364999974
Baldwin Bikes,5215751.277499983
Rowlett Bikes,867542.2436000014


Databricks visualization. Run in Databricks to view.

In [0]:
brand_sales = (
    spark.table("biker_r_project.gold.sales_fact")
    .groupBy("brand_name")
    .agg(sum("revenue").alias("revenue"))
    .orderBy(desc("revenue"))
)

display(brand_sales)

brand_name,revenue
Trek,4602754.350899985
Electra,1205320.8196000017
Surly,949507.0633000013
Sun Bicycles,341994.9275000002
Haro,185384.5536999999
Heller,171459.07569999993
Pure Cycles,149476.34000000032
Ritchey,78898.94799999999
Strider,4320.478899999999


Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.window import Window

customer_sales = (
    spark.table("biker_r_project.gold.sales_fact")
    .groupBy("customer_name")
    .agg(sum("revenue").alias("lifetime_value"))
)

display(customer_sales)

customer_name,lifetime_value
Johnathan Velazquez,10231.0464
Jaqueline Cummings,1697.9717
Joshua Robertson,1519.981
Nova Hess,13023.926
Arla Ellis,3900.0606999999995
Sharyn Hopkins,34807.93919999999
Laureen Paul,2165.0816999999997
Leslie Higgins,1372.4719
Neil Mccall,9919.3294
Alane Munoz,242.991


Databricks visualization. Run in Databricks to view.

In [0]:
orders_per_month = (
    spark.table("biker_r_project.gold.sales_fact")
    .groupBy(
        year("order_date").alias("year"),
        month("order_date").alias("month")
    )
    .agg(countDistinct("order_id").alias("orders"))
)

display(orders_per_month)

year,month,orders
2016,1,50
2016,2,49
2016,3,55
2016,4,43
2016,5,51
2016,6,45
2016,7,50
2016,8,63
2016,9,67
2016,10,64


In [0]:
inventory = spark.sql("""
SELECT
    p.product_name,
    SUM(s.quantity) stock_remaining
FROM biker_r_project.silver.products p
JOIN biker_r_project.silver.stocks s
ON p.product_id=s.product_id
GROUP BY p.product_name
""")

display(inventory)

product_name,stock_remaining
Trek Fuel EX 8 29 XT - 2018,75
Sun Bicycles Lil Kitt'n - 2017,48
Trek Ticket S Frame - 2018,56
Electra Cyclosaurus 1 (16-inch) - Boy's - 2018,32
Electra Morningstar 3i Ladies' - 2018,28
Trek Precaliber 16 Girl's - 2018,55
Electra Straight 8 1 (20-inch) - Boy's - 2018,24
Trek Remedy 9.8 - 2017,35
Trek XM700+ Lowstep - 2018,86
Electra Amsterdam Original 3i - 2015/2017,11


In [0]:
kpi = spark.sql("""
SELECT
COUNT(DISTINCT order_id) total_orders,
COUNT(DISTINCT customer_id) total_customers,
COUNT(DISTINCT product_id) total_products,
SUM(revenue) total_revenue
FROM biker_r_project.gold.sales_fact
""")

display(kpi)

total_orders,total_customers,total_products,total_revenue
1615,1445,307,7689116.5576


In [0]:
brand_sales = spark.sql("""
SELECT
    brand_name,
    ROUND(SUM(revenue),2) revenue
FROM biker_r_project.gold.sales_fact
GROUP BY brand_name
ORDER BY revenue DESC
""")

display(brand_sales)

brand_name,revenue
Trek,4602754.35
Electra,1205320.82
Surly,949507.06
Sun Bicycles,341994.93
Haro,185384.55
Heller,171459.08
Pure Cycles,149476.34
Ritchey,78898.95
Strider,4320.48


Databricks visualization. Run in Databricks to view.